In [21]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd

In [22]:
# Download GloVe embeddings if not already present
import os
if not os.path.exists("glove.6B.50d.txt"):
    !wget http://nlp.stanford.edu/data/glove.6B.zip
    !unzip -q glove.6B.zip

In [23]:
# -------------------------------
# 1. Sentences and candidates
# -------------------------------
sentences = [
    ["new","atm","opened","near","my","home","i","am","going","to","bank","to","get"],
    ["there","is","a","river","at","my","home","i","am","going","to","bank","to","get"]
]
candidates = ["money", "water"]

tokens = set(sum(sentences, []) + candidates)

# -------------------------------
# 2. Load GloVe subset
# -------------------------------
glove_file = "glove.6B.50d.txt"

def load_glove_subset(path, tokens):
    tokens = set(tokens)
    emb = {}
    with open(path, encoding="utf8") as f:
        for line in f:
            word, *vals = line.split()
            if word in tokens:
                emb[word] = np.array(vals, dtype=float)
                if len(emb) == len(tokens):
                    break
    return emb


emb = load_glove_subset(glove_file, tokens)


# -------------------------------
# 3. Create input matrices
# -------------------------------
def make_matrix(sentence):
    return torch.tensor(np.vstack([emb[word] for word in sentence]), dtype=torch.float32)

X_list = [make_matrix(s) for s in sentences]
query_positions = [s.index("bank") for s in sentences]
targets = torch.tensor([0, 1])
cand_mat = torch.tensor(np.vstack([emb[w] for w in candidates]), dtype=torch.float32)

print(query_positions)

[10, 11]


In [38]:
print(sentences[0][10])
print(sentences[1][11])

bank
bank


Scores with just glove vector bank

In [39]:
bank_vec = torch.tensor(emb["bank"], dtype=torch.float32)

for i, sentence in enumerate(sentences):
    print(f"\nSentence {i+1}:")
    scores = [(word, torch.dot(bank_vec, torch.tensor(emb[word], dtype=torch.float32)).item()) for word in sentence]
    for word, score in sorted(scores, key=lambda x: -x[1]):
        print(f"  {word:<10} {score:.4f}")


Sentence 1:
  bank       36.3920
  to         19.2234
  to         19.2234
  new        17.4814
  near       17.0214
  opened     16.2964
  home       15.0630
  get        14.3078
  going      13.9664
  atm        11.4918
  i          11.2500
  my         10.6286
  am         9.9148

Sentence 2:
  bank       36.3920
  to         19.2234
  to         19.2234
  at         19.0401
  river      16.6162
  a          16.4929
  is         15.3286
  home       15.0630
  get        14.3078
  there      14.0527
  going      13.9664
  i          11.2500
  my         10.6286
  am         9.9148


### Creating X

 Query (Q) = what we’re focusing on right now — the thing that wants information.

 Keys (K) = the things we might look at — each represents something that could be relevant.

 Values (V) = the actual information we’ll retrieve if we decide a key is relevant.

 We want the model to understand what sense of “bank” we’re using — is it a financial institution or river bank?
The word “bank” is the ambiguous word.
That’s why we treat it as the query — it’s the word asking for context.


 The query “bank” asks its neighbors: “Who among you can tell me what kind of bank I am?”
Then the attention scores decide which context words (keys) to pay attention to.

In [24]:
print(X_list[0].shape)#sentence 1
print(X_list[1].shape)#sentence 2

torch.Size([13, 50])
torch.Size([14, 50])


In [25]:

# -------------------------------
# 4. Define trainable weights
# -------------------------------
D_in, D_k, D_v = 50, 32, 32
W_Q = nn.Parameter(torch.randn(D_in, D_k) * 0.01)
W_K = nn.Parameter(torch.randn(D_in, D_k) * 0.01)
W_V = nn.Parameter(torch.randn(D_in, D_v) * 0.01)

# -------------------------------
# 5. Forward attention function
# -------------------------------
def forward_attention(X, q_pos, cand_mat, W_Q, W_K, W_V):
    Q = X[q_pos].unsqueeze(0) @ W_Q
    K = X @ W_K
    V = X @ W_V
    scores = Q @ K.T / torch.sqrt(torch.tensor(K.shape[1], dtype=torch.float32))
    alpha = torch.softmax(scores, dim=-1)
    context = alpha @ V
    logits = context @ (cand_mat @ W_V).T
    return logits, alpha


In [26]:
# -------------------------------
# 6. Optimizer and loss
# -------------------------------
optimizer = optim.Adam([W_Q, W_K, W_V], lr=0.05)
loss_fn = nn.CrossEntropyLoss()
epochs = 5

# -------------------------------
# 7. Training loop
# -------------------------------

print("Training to predict: Sentence 1 -> 'money', Sentence 2 -> 'water'\n")
for epoch in range(epochs):
    optimizer.zero_grad()
    logits_batch = torch.cat([forward_attention(X_list[i], query_positions[i], cand_mat, W_Q, W_K, W_V)[0]
                              for i in range(len(X_list))], dim=0)
    loss = loss_fn(logits_batch, targets)
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Training to predict: Sentence 1 -> 'money', Sentence 2 -> 'water'

Epoch 0, Loss: 0.6932
Epoch 1, Loss: 0.0381
Epoch 2, Loss: 16.0655
Epoch 3, Loss: 0.0000
Epoch 4, Loss: 0.0000


In [27]:
# -------------------------------
# 8. Evaluation
# -------------------------------
print("\n" + "="*70)
print("EVALUATION: Word Sense Disambiguation")
print("="*70)

with torch.no_grad():
    for i, X in enumerate(X_list):
        logits, alpha = forward_attention(X, query_positions[i], cand_mat, W_Q, W_K, W_V)
        probs = torch.softmax(logits, dim=-1)
        sent_tokens = sentences[i]

        print(f"Sentence {i+1}: {' '.join(sentences[i])}")
        print(f"Target word: '{candidates[targets[i]]}'")
        print("Predictions:")

        for j, cand in enumerate(candidates):
            marker = "✓" if j == targets[i] else "✗"
            print(f"  {marker} {cand:10s}: {probs[0,j].item():.4f}")

        df = pd.DataFrame({"token": sent_tokens, "attention": alpha.squeeze().numpy()})
        top_attention = df.sort_values('attention', ascending=False).head(10)
        print("\n⚡ Top attention tokens:")

        for _, row in top_attention.iterrows():
            print(f"  {row['token']:10s}: {row['attention']:.4f}")
        print("-"*70)


EVALUATION: Word Sense Disambiguation
Sentence 1: new atm opened near my home i am going to bank to get
Target word: 'money'
Predictions:
  ✓ money     : 1.0000
  ✗ water     : 0.0000

⚡ Top attention tokens:
  bank      : 0.9998
  near      : 0.0001
  going     : 0.0000
  to        : 0.0000
  to        : 0.0000
  get       : 0.0000
  new       : 0.0000
  my        : 0.0000
  i         : 0.0000
  opened    : 0.0000
----------------------------------------------------------------------
Sentence 2: there is a river at my home i am going to bank to get
Target word: 'water'
Predictions:
  ✗ money     : 0.0000
  ✓ water     : 1.0000

⚡ Top attention tokens:
  river     : 1.0000
  bank      : 0.0000
  going     : 0.0000
  to        : 0.0000
  to        : 0.0000
  at        : 0.0000
  is        : 0.0000
  get       : 0.0000
  there     : 0.0000
  my        : 0.0000
----------------------------------------------------------------------


scores with context vector bank

In [40]:
with torch.no_grad():
    for i, X in enumerate(X_list):
        logits, alpha = forward_attention(X, query_positions[i], cand_mat, W_Q, W_K, W_V)
        context = alpha @ (X @ W_V)  # the new bank representation

        print(f"\nSentence {i+1}:")
        V_all = X @ W_V
        scores = [(sentences[i][j], torch.dot(context.squeeze(), V_all[j]).item()) for j in range(len(sentences[i]))]
        for word, score in sorted(scores, key=lambda x: -x[1]):
            print(f"  {word:<10} {score:.4f}")


Sentence 1:
  atm        6.7496
  bank       4.9799
  new        2.1267
  home       1.7654
  opened     1.0378
  am         -0.7427
  to         -0.9059
  to         -0.9059
  get        -1.3276
  going      -1.7719
  i          -4.8636
  my         -5.7445
  near       -6.5813

Sentence 2:
  river      141.6171
  my         51.7514
  i          43.3600
  is         42.9301
  there      28.1295
  going      25.0329
  a          24.2760
  get        17.9055
  to         17.8604
  to         17.8604
  at         12.9315
  am         7.8342
  home       0.6676
  bank       -17.4864
